# 2024 영등포구청역 비교

- 주차장: 영등포구청역 공영주차장
- 지하철: `analystics_sub_new_24.ipynb` 기준 여의도·여의나루
- 교통량: `analystics_traffic_new_24.ipynb` 기준 서강대교, 마포대교, 원효대교
- 날짜 구분: 평일 / 주말(토, 일)
- 시간 구간: 06시~23시
- 비교값: 각 소스별 입/출 중 더 큰 값의 시간대 중앙값

In [ ]:
from pathlib import Path
import platform
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

warnings.filterwarnings("ignore")

project_root = Path.cwd().resolve()
if not (project_root / "Data").exists() and (project_root.parent / "Data").exists():
    project_root = project_root.parent

if platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "Malgun Gothic"

plt.rcParams["axes.unicode_minus"] = False

analysis_year = 2024
focused_hours = list(range(6, 24))
selected_bridges = ["서강대교", "마포대교", "원효대교"]
selected_stations = ["여의도", "여의나루"]
target_parking_name = "영등포구청역"

parking_path = project_root / "Data/testset/서울시설공단_공영주차장 시간별 주차현황_20250531.csv"
metro_path = project_root / "Data/Metro/서울교통공사_역별 시간대별 승하차인원(24.1~24.12) (1).csv"
traffic_path = project_root / "Data/Trafficdata/traffic_24.csv"


def minmax_scale_series(series):
    series = series.astype(float)
    min_value = series.min()
    max_value = series.max()
    if pd.isna(min_value) or pd.isna(max_value) or min_value == max_value:
        return pd.Series(0.0, index=series.index, dtype=float)
    return (series - min_value) / (max_value - min_value)

In [ ]:
parking_df = pd.read_csv(parking_path, encoding="euc-kr")
parking_df["시간대"] = pd.to_datetime(parking_df["시간대"])
parking_df = parking_df[parking_df["주차장명"] == target_parking_name].copy()
parking_df = parking_df[parking_df["시간대"].dt.year == analysis_year].copy()
parking_df["hour"] = parking_df["시간대"].dt.hour
parking_df["day_type"] = np.where(parking_df["시간대"].dt.dayofweek >= 5, "주말", "평일")
parking_df["parking_max"] = parking_df[["입차대수", "출차대수"]].max(axis=1)

parking_profile = (
    parking_df[parking_df["hour"].between(6, 23)]
    .groupby(["day_type", "hour"], as_index=False)["parking_max"]
    .median()
)

display(parking_profile.head())

In [ ]:
metro_df = pd.read_csv(metro_path, encoding="cp949")
station_col = "역명"
direction_col = "승하차구분" if "승하차구분" in metro_df.columns else "구분"
date_col = "수송일자" if "수송일자" in metro_df.columns else "날짜"
metro_hour_cols = [
    f"{hour:02d}시-{hour + 1:02d}시"
    for hour in focused_hours
    if f"{hour:02d}시-{hour + 1:02d}시" in metro_df.columns
]

metro_df[date_col] = pd.to_datetime(metro_df[date_col])
metro_df["날짜타입"] = metro_df[date_col].dt.weekday.map(lambda x: "주말" if x >= 5 else "평일")
filtered_metro_df = metro_df[metro_df[station_col].isin(selected_stations)].copy()
cleaned_metro_df = filtered_metro_df[[station_col, date_col, "날짜타입", direction_col] + metro_hour_cols].copy()

station_avg_df = (
    cleaned_metro_df
    .groupby([station_col, "날짜타입", direction_col], as_index=False)[metro_hour_cols]
    .mean()
)

metro_median_df = station_avg_df.melt(
    id_vars=[station_col, "날짜타입", direction_col],
    value_vars=metro_hour_cols,
    var_name="시간",
    value_name="평균값",
)
time_order_map = {column: idx for idx, column in enumerate(metro_hour_cols)}
metro_median_df["시간_순서"] = metro_median_df["시간"].map(time_order_map)
metro_median_df = (
    metro_median_df
    .groupby(["날짜타입", direction_col, "시간", "시간_순서"], as_index=False)["평균값"]
    .median()
    .rename(columns={"평균값": "중앙값"})
    .sort_values(["날짜타입", direction_col, "시간_순서"])
    .reset_index(drop=True)
)

metro_profile = metro_median_df.pivot(
    index=["날짜타입", "시간", "시간_순서"],
    columns=direction_col,
    values="중앙값",
).reset_index()
metro_dir_cols = [col for col in metro_profile.columns if col not in ["날짜타입", "시간", "시간_순서"]]
metro_profile["subway_max"] = metro_profile[metro_dir_cols].max(axis=1)
metro_profile["hour"] = metro_profile["시간"].str.extract(r"^(\d{2})").astype(int)
metro_profile = metro_profile[["날짜타입", "hour", "subway_max"]].rename(columns={"날짜타입": "day_type"})

display(metro_profile.head())

In [ ]:
traffic_df = pd.read_csv(traffic_path)
traffic_df["일자"] = pd.to_datetime(traffic_df["일자"].astype(str), format="%Y%m%d")
traffic_df["날짜타입"] = traffic_df["일자"].dt.weekday.map(lambda x: "주말" if x >= 5 else "평일")
traffic_hour_cols = [f"{hour}시" for hour in focused_hours if f"{hour}시" in traffic_df.columns]

filtered_traffic_df = traffic_df[traffic_df["지점명"].isin(selected_bridges)].copy()
cleaned_traffic_df = filtered_traffic_df[["지점명", "일자", "날짜타입", "방향"] + traffic_hour_cols].copy()

station_avg_df = (
    cleaned_traffic_df
    .groupby(["지점명", "날짜타입", "방향"], as_index=False)[traffic_hour_cols]
    .mean()
)

traffic_median_df = station_avg_df.melt(
    id_vars=["지점명", "날짜타입", "방향"],
    value_vars=traffic_hour_cols,
    var_name="시간",
    value_name="평균값",
)
traffic_median_df["시간_순서"] = traffic_median_df["시간"].str.extract(r"(\d+)").astype(int)
traffic_median_df = (
    traffic_median_df
    .groupby(["날짜타입", "방향", "시간", "시간_순서"], as_index=False)["평균값"]
    .median()
    .rename(columns={"평균값": "중앙값"})
    .sort_values(["날짜타입", "방향", "시간_순서"])
    .reset_index(drop=True)
)

traffic_profile = traffic_median_df.pivot(
    index=["날짜타입", "시간", "시간_순서"],
    columns="방향",
    values="중앙값",
).reset_index()
traffic_dir_cols = [col for col in traffic_profile.columns if col not in ["날짜타입", "시간", "시간_순서"]]
traffic_profile["traffic_max"] = traffic_profile[traffic_dir_cols].max(axis=1)
traffic_profile["hour"] = traffic_profile["시간"].str.extract(r"(\d+)").astype(int)
traffic_profile = traffic_profile[["날짜타입", "hour", "traffic_max"]].rename(columns={"날짜타입": "day_type"})

display(traffic_profile.head())

In [ ]:
compare_df = (
    parking_profile
    .merge(metro_profile, on=["day_type", "hour"], how="inner")
    .merge(traffic_profile, on=["day_type", "hour"], how="inner")
    .sort_values(["day_type", "hour"])
    .reset_index(drop=True)
)
compare_df["combined_raw"] = compare_df["subway_max"] + compare_df["traffic_max"] / 2.0

scaled_parts = []
corr_rows = []

for day_type, group in compare_df.groupby("day_type", sort=False):
    scaled_group = group.sort_values("hour").copy()
    scaled_group["parking_scaled"] = minmax_scale_series(scaled_group["parking_max"]).values
    scaled_group["subway_scaled"] = minmax_scale_series(scaled_group["subway_max"]).values
    scaled_group["traffic_scaled"] = minmax_scale_series(scaled_group["traffic_max"]).values
    scaled_group["combined_scaled"] = minmax_scale_series(scaled_group["combined_raw"]).values
    scaled_parts.append(scaled_group)
    corr_rows.append({
        "day_type": day_type,
        "corr_subway_vs_parking": scaled_group["subway_scaled"].corr(scaled_group["parking_scaled"]),
        "corr_traffic_vs_parking": scaled_group["traffic_scaled"].corr(scaled_group["parking_scaled"]),
        "corr_combined_vs_parking": scaled_group["combined_scaled"].corr(scaled_group["parking_scaled"]),
    })

compare_scaled_df = pd.concat(scaled_parts, ignore_index=True)
corr_df = pd.DataFrame(corr_rows)

fig, axes = plt.subplots(1, 2, figsize=(16, 6), sharey=True)
plot_order = ["평일", "주말"]
plot_colors = {
    "subway_scaled": "#1f77b4",
    "traffic_scaled": "#d62728",
    "combined_scaled": "#9467bd",
    "parking_scaled": "#2ca02c",
}
plot_labels = {
    "subway_scaled": "지하철",
    "traffic_scaled": "교통량",
    "combined_scaled": "지하철 + 교통량/2",
    "parking_scaled": "영등포구청역 주차장",
}

for ax, day_type in zip(axes, plot_order):
    subset = compare_scaled_df[compare_scaled_df["day_type"] == day_type].sort_values("hour")
    for col in ["subway_scaled", "traffic_scaled", "combined_scaled", "parking_scaled"]:
        ax.plot(
            subset["hour"],
            subset[col],
            marker="o",
            linewidth=2.5 if col == "parking_scaled" else 2.0,
            color=plot_colors[col],
            label=plot_labels[col],
        )

    corr_row = corr_df[corr_df["day_type"] == day_type].iloc[0]
    corr_text = "\n".join([
        f"corr(주차장, 지하철) = {corr_row['corr_subway_vs_parking']:.3f}",
        f"corr(주차장, 교통량) = {corr_row['corr_traffic_vs_parking']:.3f}",
        f"corr(주차장, 지하철+교통량/2) = {corr_row['corr_combined_vs_parking']:.3f}",
    ])

    ax.set_title(day_type)
    ax.set_xlabel("시간")
    ax.set_xticks(focused_hours)
    ax.set_ylim(0, 1)
    ax.grid(alpha=0.3)
    ax.legend()
    ax.text(
        0.5,
        -0.28,
        corr_text,
        transform=ax.transAxes,
        ha="center",
        va="top",
        fontsize=10,
    )

axes[0].set_ylabel("Min-Max Scaled Value")
fig.suptitle("2024년 영등포구청역 주차장 vs 지하철/교통량 비교")
plt.tight_layout(rect=[0, 0.08, 1, 0.96])
plt.show()

display(compare_scaled_df)
corr_df

In [ ]:
weekday_shift_df = compare_scaled_df[compare_scaled_df["day_type"] == "평일"].sort_values("hour").copy()
weekday_shift_df["subway_shifted"] = weekday_shift_df["subway_scaled"].shift(1)
weekday_shift_df["traffic_shifted"] = weekday_shift_df["traffic_scaled"].shift(1)
weekday_shift_df["combined_shifted"] = weekday_shift_df["combined_scaled"].shift(1)
weekday_shift_df = weekday_shift_df[weekday_shift_df["hour"] >= 7].copy()

shift_corr_df = pd.DataFrame([
    {"series": "지하철(우측 1칸 shift)", "corr": weekday_shift_df["subway_shifted"].corr(weekday_shift_df["parking_scaled"])},
    {"series": "교통량(우측 1칸 shift)", "corr": weekday_shift_df["traffic_shifted"].corr(weekday_shift_df["parking_scaled"])},
    {"series": "지하철 + 교통량/2(우측 1칸 shift)", "corr": weekday_shift_df["combined_shifted"].corr(weekday_shift_df["parking_scaled"])},
])

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(
    weekday_shift_df["hour"],
    weekday_shift_df["subway_shifted"],
    marker="o",
    linewidth=2,
    color="#1f77b4",
    label="지하철 shifted",
)
ax.plot(
    weekday_shift_df["hour"],
    weekday_shift_df["traffic_shifted"],
    marker="o",
    linewidth=2,
    color="#d62728",
    label="교통량 shifted",
)
ax.plot(
    weekday_shift_df["hour"],
    weekday_shift_df["combined_shifted"],
    marker="o",
    linewidth=2,
    color="#9467bd",
    label="지하철 + 교통량/2 shifted",
)
ax.plot(
    weekday_shift_df["hour"],
    weekday_shift_df["parking_scaled"],
    marker="o",
    linewidth=2.8,
    color="#2ca02c",
    label="영등포구청역 주차장",
)

corr_text = "\n".join(
    f"corr(주차장, {row['series']}) = {row['corr']:.3f}"
    for _, row in shift_corr_df.iterrows()
)

ax.set_title("평일 비교 (지하철/교통량/결합 우측 1칸 shift, 07-23시)")
ax.set_xlabel("시간")
ax.set_ylabel("Min-Max Scaled Value")
ax.set_xticks(range(7, 24))
ax.set_ylim(0, 1)
ax.grid(alpha=0.3)
ax.legend()
ax.text(
    0.5,
    -0.24,
    corr_text,
    transform=ax.transAxes,
    ha="center",
    va="top",
    fontsize=10,
)

plt.tight_layout(rect=[0, 0.08, 1, 0.98])
plt.show()

shift_corr_df